# Week 3: Baseline Rules & First Insights

## Goals:
- Create simple rules to catch issues
- Implement tolerance rule for repark errors
- Test greedy batching vs random assignment
- Compute availability-adjusted picks/hour
- Measure CPS release→arrival delays

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load Feature Data

In [ ]:
# Load the feature tables created in Week 2
event_features = pd.read_parquet('../data/event_features.parquet')
order_features = pd.read_parquet('../data/order_features.parquet')
station_hourly_features = pd.read_parquet('../data/station_hourly_features.parquet')

print("Feature tables loaded successfully!")
print(f"Event features shape: {event_features.shape}")
print(f"Order features shape: {order_features.shape}")
print(f"Station hourly features shape: {station_hourly_features.shape}")

In [ ]:
df = pd.read_parquet("../data/event_features.parquet", engine='pyarrow')
print(df.shape)
print(df.head())


In [ ]:
# 1. Avg latency per operator
latency = df.groupby("operator_id")["latency_sec"].mean().sort_values()
latency.plot(kind="barh", title="Average Latency per Operator")
plt.show()

# 2. Walking distance distribution
df["distance_walked"].hist(bins=50)
plt.title("Distribution of Walking Distance")
plt.xlabel("Distance walked")
plt.ylabel("Count")
plt.show()

# 3. CPS delay per wave
if "cps_delay_minutes" in df.columns:
    cps = df.groupby("wave_id")["cps_delay_minutes"].mean()
    cps.plot(kind="line", title="Avg CPS Delay per Wave")
    plt.ylabel("Delay (minutes)")
    plt.show()

# 4. Repark errors
if "is_repark_error" in df.columns:
    errors = df.groupby("operator_id")["is_repark_error"].mean().sort_values()
    errors.plot(kind="barh", title="Repark Error Rate per Operator")
    plt.show()

## 2. Implement Tolerance Rule for Repark Errors

### Tolerance rule: |residual| > a + b√qty + c%

In [ ]:
# Define tolerance parameters (these would be determined based on domain knowledge)
a = 0.5  # Base tolerance
b = 0.1  # Coefficient for square root of quantity
c = 0.02  # Percentage coefficient (2%)

# Calculate tolerance threshold for each event
event_features['tolerance_threshold'] = a + b * np.sqrt(event_features['quantity']) + c * event_features['quantity']

# Identify repark errors based on tolerance rule
event_features['is_repark_error'] = np.abs(event_features['residual']) > event_features['tolerance_threshold']

# Calculate error rate
error_rate = event_features['is_repark_error'].mean()
print(f"Overall repark error rate: {error_rate:.2%}")

# Error rate by SKU
sku_error_rates = event_features.groupby('sku_id')['is_repark_error'].agg(['count', 'sum', 'mean']).reset_index()
sku_error_rates.columns = ['sku_id', 'total_events', 'error_count', 'error_rate']
sku_error_rates = sku_error_rates[sku_error_rates['total_events'] >= 10]  # Filter for SKUs with sufficient events
sku_error_rates = sku_error_rates.sort_values('error_rate', ascending=False)

print("\nTop 10 SKUs with highest error rates:")
print(sku_error_rates.head(10))

In [ ]:
# Visualize error rates by SKU
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(sku_error_rates['error_rate'], bins=30, edgecolor='black')
plt.title('Distribution of SKU Error Rates')
plt.xlabel('Error Rate')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
top_skus = sku_error_rates.head(20)
plt.bar(range(len(top_skus)), top_skus['error_rate'])
plt.title('Top 20 SKUs by Error Rate')
plt.xlabel('SKU Rank')
plt.ylabel('Error Rate')
plt.xticks(range(0, len(top_skus), 2), range(1, len(top_skus)+1, 2))

plt.tight_layout()
plt.show()

## 3. Test Greedy Batching vs Random Assignment

### Group orders by SKU overlap

In [ ]:
# For batching, we'll simulate orders with SKUs
# In a real scenario, we would have detailed location data

# Create a simplified order-SKU matrix
order_skus = event_features[['order_id', 'sku_id']].drop_duplicates()

# Count SKUs per order
skus_per_order = order_skus.groupby('order_id').size().reset_index(name='sku_count')

# Count orders per SKU
orders_per_sku = order_skus.groupby('sku_id').size().reset_index(name='order_count')

print(f"Average SKUs per order: {skus_per_order['sku_count'].mean():.2f}")
print(f"Average orders per SKU: {orders_per_sku['order_count'].mean():.2f}")

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

# Simulate greedy batching algorithm - optimized version
def greedy_batching(orders, order_skus, batch_size=10):
    """Optimized greedy batching based on SKU overlap"""
    
    # Pre-process order-SKU relationships into dictionary for faster lookup
    order_sku_dict = defaultdict(set)
    for _, row in order_skus.iterrows():
        order_sku_dict[row['order_id']].add(row['sku_id'])
    
    # Optimized Jaccard similarity function
    def jaccard_similarity(skus1, skus2):
        intersection = len(skus1.intersection(skus2))
        union = len(skus1.union(skus2))
        return intersection / union if union > 0 else 0
    
    # Simple greedy approach: pick orders with highest overlap
    orders_list = list(order_sku_dict.keys())
    batches = []
    used_orders = set()
    
    for order in orders_list:
        if order in used_orders:
            continue
            
        # Create a batch starting with this order
        batch = [order]
        used_orders.add(order)
        order_skus_set = order_sku_dict[order]
        
        # Add similar orders to the batch (limit search for performance)
        similarities = []
        search_limit = min(100, len(orders_list))  # Limit search to improve performance
        
        for i, other_order in enumerate(orders_list):
            if i >= search_limit:
                break
            if other_order not in used_orders and other_order != order:
                other_skus_set = order_sku_dict[other_order]
                sim = jaccard_similarity(order_skus_set, other_skus_set)
                if sim > 0:  # Only consider orders with some similarity
                    similarities.append((other_order, sim))
        
        # Sort by similarity and add to batch
        similarities.sort(key=lambda x: x[1], reverse=True)
        for other_order, sim in similarities[:batch_size-1]:
            if len(batch) < batch_size:
                batch.append(other_order)
                used_orders.add(other_order)
        
        batches.append(batch)
    
    return batches

# Simulate random batching for comparison
def random_batching(orders, batch_size=10):
    """Random batching"""
    orders_copy = orders.copy()
    np.random.shuffle(orders_copy)
    batches = [orders_copy[i:i + batch_size] for i in range(0, len(orders_copy), batch_size)]
    return batches

# MAIN EXECUTION - Limited to 10,000 orders
print("Starting warehouse optimization with limited dataset...")

# Limit to first 10,000 unique orders for performance
unique_orders = order_skus['order_id'].unique()[:10000]
print(f"Total orders available: {len(order_skus['order_id'].unique())}")
print(f"Using subset of: {len(unique_orders)} orders")

# Filter order_skus to only include the selected orders
filtered_order_skus = order_skus[order_skus['order_id'].isin(unique_orders)]

# Apply batching algorithms
print("Running greedy batching algorithm...")
greedy_batches = greedy_batching(unique_orders, filtered_order_skus)

print("Running random batching algorithm...")
random_batches = random_batching(unique_orders.copy())

# Results
print("\n" + "="*50)
print("BATCHING RESULTS")
print("="*50)
print(f"Number of orders processed: {len(unique_orders):,}")
print(f"Number of greedy batches: {len(greedy_batches):,}")
print(f"Number of random batches: {len(random_batches):,}")
print(f"Average batch size (greedy): {np.mean([len(b) for b in greedy_batches]):.2f}")
print(f"Average batch size (random): {np.mean([len(b) for b in random_batches]):.2f}")

# Additional performance metrics
print("\n" + "="*30)
print("PERFORMANCE METRICS")
print("="*30)
total_skus_greedy = sum(len(filtered_order_skus[filtered_order_skus['order_id'].isin(batch)]['sku_id'].unique()) for batch in greedy_batches)
total_skus_random = sum(len(filtered_order_skus[filtered_order_skus['order_id'].isin(batch)]['sku_id'].unique()) for batch in random_batches)

print(f"Average unique SKUs per batch (greedy): {total_skus_greedy/len(greedy_batches):.2f}")
print(f"Average unique SKUs per batch (random): {total_skus_random/len(random_batches):.2f}")

# Sample batch analysis
if len(greedy_batches) > 0:
    sample_batch = greedy_batches[0]
    sample_skus = filtered_order_skus[filtered_order_skus['order_id'].isin(sample_batch)]['sku_id'].unique()
    print(f"\nSample greedy batch size: {len(sample_batch)} orders, {len(sample_skus)} unique SKUs")

In [ ]:
# Get unique orders
unique_orders = order_skus['order_id'].unique()

# Apply batching algorithms - MODIFIED TO USE ONLY 1000 ORDERS
limited_orders = unique_orders[:1000]
limited_order_skus = order_skus[order_skus['order_id'].isin(limited_orders)]

print("=== RUNNING ALGORITHMS ===")
greedy_batches = greedy_batching(limited_orders, limited_order_skus)
random_batches = random_batching(limited_orders.copy())

print(f"Number of orders: {len(limited_orders)}")
print(f"Number of greedy batches: {len(greedy_batches)}")
print(f"Number of random batches: {len(random_batches)}")
print(f"Average batch size (greedy): {np.mean([len(b) for b in greedy_batches]):.2f}")
print(f"Average batch size (random): {np.mean([len(b) for b in random_batches]):.2f}")

print("\n=== DETAILED COMPARISON ===")
print("First 5 greedy batches:")
for i, batch in enumerate(greedy_batches[:5]):
    print(f"  Batch {i+1}: {batch}")

print("\nFirst 5 random batches:")
for i, batch in enumerate(random_batches[:5]):
    print(f"  Batch {i+1}: {batch}")

# Calculate SKU overlap efficiency
def calculate_sku_overlap(batches, order_skus_data):
    total_overlap = 0
    total_possible = 0
    
    for batch in batches:
        if len(batch) > 1:
            batch_skus = []
            for order_id in batch:
                order_skus_list = order_skus_data[order_skus_data['order_id'] == order_id]['sku_id'].tolist()
                batch_skus.append(set(order_skus_list))
            
            # Calculate pairwise overlaps
            overlaps = 0
            comparisons = 0
            for i in range(len(batch_skus)):
                for j in range(i+1, len(batch_skus)):
                    overlap = len(batch_skus[i].intersection(batch_skus[j]))
                    total_size = len(batch_skus[i].union(batch_skus[j]))
                    if total_size > 0:
                        overlaps += overlap / total_size
                    comparisons += 1
            
            if comparisons > 0:
                total_overlap += overlaps / comparisons
                total_possible += 1
    
    return total_overlap / total_possible if total_possible > 0 else 0

print("\n=== ALGORITHM EFFECTIVENESS ===")
greedy_overlap = calculate_sku_overlap(greedy_batches, limited_order_skus)
random_overlap = calculate_sku_overlap(random_batches, limited_order_skus)

print(f"Average SKU overlap (greedy): {greedy_overlap:.4f}")
print(f"Average SKU overlap (random): {random_overlap:.4f}")
print(f"Improvement ratio: {greedy_overlap/random_overlap:.2f}x" if random_overlap > 0 else "Cannot calculate ratio")

# Show that the algorithms ARE working differently
print("\n=== VERIFICATION ===")
identical_batches = 0
for i in range(min(len(greedy_batches), len(random_batches))):
    if set(greedy_batches[i]) == set(random_batches[i]):
        identical_batches += 1

print(f"Identical batches between algorithms: {identical_batches}/{len(greedy_batches)}")
if identical_batches == 0:
    print("✓ Algorithms are producing different results (working correctly!)")
else:
    print("⚠ Some batches are identical (may indicate an issue)")

## 4. Compute Availability-Adjusted Picks/Hour

### Exclude waiting/walking time

In [ ]:
# Calculate picks per hour for each operator
operator_performance = event_features.groupby('operator_id').agg({
    'order_id': 'nunique',  # Number of orders
    'sku_id': 'count',  # Number of picks
    'timestamp': ['min', 'max'],  # Time range
    'latency_sec': 'mean'  # Average latency
}).reset_index()

# Flatten column names
operator_performance.columns = ['operator_id', 'orders_count', 'picks_count', 'start_time', 'end_time', 'avg_latency']

# Calculate working time in hours
operator_performance['working_time_hours'] = (operator_performance['end_time'] - operator_performance['start_time']).dt.total_seconds() / 3600

# Calculate raw picks per hour
operator_performance['raw_picks_per_hour'] = operator_performance['picks_count'] / operator_performance['working_time_hours']

# Adjust for latency (waiting/walking time)
# Assume latency represents non-productive time
operator_performance['productive_time_hours'] = operator_performance['working_time_hours'] - \
    (operator_performance['avg_latency'] * operator_performance['picks_count'] / 3600)

# Ensure we don't have negative productive time
operator_performance['productive_time_hours'] = np.maximum(operator_performance['productive_time_hours'], 0.1)

# Calculate availability-adjusted picks per hour
operator_performance['adjusted_picks_per_hour'] = operator_performance['picks_count'] / operator_performance['productive_time_hours']

print("Operator Performance Metrics:")
print(operator_performance[['operator_id', 'raw_picks_per_hour', 'adjusted_picks_per_hour']])

In [ ]:
# Visualize performance comparison
plt.figure(figsize=(12, 6))
x = np.arange(len(operator_performance))
width = 0.35

plt.bar(x - width/2, operator_performance['raw_picks_per_hour'], width, label='Raw Picks/Hour')
plt.bar(x + width/2, operator_performance['adjusted_picks_per_hour'], width, label='Adjusted Picks/Hour')

plt.xlabel('Operator')
plt.ylabel('Picks per Hour')
plt.title('Operator Performance: Raw vs Adjusted Picks per Hour')
plt.xticks(x, operator_performance['operator_id'], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Measure CPS Release→Arrival Delays

### (release time vs infeed time)

Note: Since we don't have explicit CPS data in our datasets, we'll simulate this based on wave processing times.

In [ ]:
# Simulate CPS delays based on wave processing
# Assume wave creation time as "release time" and first pick time as "arrival time"

wave_times = event_features.groupby('wave_id').agg({
    'timestamp': ['min', 'max']
}).reset_index()

# Flatten column names
wave_times.columns = ['wave_id', 'first_pick_time', 'last_pick_time']

# Simulate release time (some time before first pick)
wave_times['release_time'] = wave_times['first_pick_time'] - pd.to_timedelta(np.random.exponential(300, len(wave_times)), unit='s')  # 5 min avg delay

# Calculate CPS delay
wave_times['cps_delay_seconds'] = (wave_times['first_pick_time'] - wave_times['release_time']).dt.total_seconds()

# Remove negative delays (which shouldn't happen in reality)
wave_times = wave_times[wave_times['cps_delay_seconds'] >= 0]

print("CPS Delay Statistics:")
print(wave_times['cps_delay_seconds'].describe())

# Distribution of delays
plt.figure(figsize=(10, 6))
plt.hist(wave_times['cps_delay_seconds'], bins=50, edgecolor='black')
plt.xlabel('CPS Delay (seconds)')
plt.ylabel('Frequency')
plt.title('Distribution of CPS Release→Arrival Delays')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# First, let's check what columns are available
print("=== DATAFRAME INSPECTION ===")
print("DataFrame shape:", event_features.shape)
print("\nAvailable columns:")
print(event_features.columns.tolist())

print("\nColumn data types:")
print(event_features.dtypes)

# Look for columns that might be related to errors
error_related_columns = [col for col in event_features.columns if 'error' in col.lower()]
repark_related_columns = [col for col in event_features.columns if 'repark' in col.lower()]

print(f"\nColumns containing 'error': {error_related_columns}")
print(f"Columns containing 'repark': {repark_related_columns}")

# Check first few rows to understand the data structure
print("\nFirst 5 rows of the dataframe:")
print(event_features.head())

# Let's also check if there are any columns that might indicate errors
print("\nLooking for potential error indicator columns...")
potential_error_cols = [col for col in event_features.columns if any(keyword in col.lower() 
                       for keyword in ['error', 'fail', 'mistake', 'wrong', 'repark', 'issue'])]
print(f"Potential error-related columns: {potential_error_cols}")

# If we find the correct column name, we can proceed with the analysis
print("\n" + "="*50)
print("SUGGESTED FIXES:")
print("="*50)

if error_related_columns:
    suggested_col = error_related_columns[0]
    print(f"1. Try using '{suggested_col}' instead of 'is_repark_error'")
elif repark_related_columns:
    suggested_col = repark_related_columns[0]
    print(f"1. Try using '{suggested_col}' instead of 'is_repark_error'")
elif potential_error_cols:
    suggested_col = potential_error_cols[0]
    print(f"1. Try using '{suggested_col}' instead of 'is_repark_error'")
else:
    print("1. No obvious error columns found. You may need to:")
    print("   - Create an error indicator column")
    print("   - Check if the column name is spelled differently")
    print("   - Verify you're using the correct DataFrame")

# Template code with placeholder for correct column name
print(f"\n2. Here's the template code - replace 'CORRECT_ERROR_COLUMN' with the right column name:")
print("""
# Error rates by SKU/operator (TEMPLATE)
if 'CORRECT_ERROR_COLUMN' in event_features.columns:
    error_by_sku_station = event_features.groupby(['sku_id', 'operator_id'])['CORRECT_ERROR_COLUMN'].mean().reset_index()
    error_by_sku_station = error_by_sku_station[error_by_sku_station['CORRECT_ERROR_COLUMN'] > 0]
    
    plt.figure(figsize=(12, 8))
    pivot_data = error_by_sku_station.pivot(index='sku_id', columns='operator_id', values='CORRECT_ERROR_COLUMN')
    sns.heatmap(pivot_data.head(20).fillna(0), annot=True, fmt='.2f', cmap='Reds')
    plt.title('Error Rates by SKU and Operator (Top 20 SKUs)')
    plt.xlabel('Operator')
    plt.ylabel('SKU')
    plt.tight_layout()
    plt.show()
else:
    print("Error column not found!")
""")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("=== CREATING ERROR INDICATORS ===")

# Option 1: High latency as error indicator
# Define high latency as > 95th percentile
latency_threshold = event_features['latency_sec'].quantile(0.95)
print(f"95th percentile latency: {latency_threshold:.2f} seconds")

# Option 2: High residual as error indicator  
# Define high residual as > 95th percentile
residual_threshold = event_features['residual'].quantile(0.95)
print(f"95th percentile residual: {residual_threshold:.2f}")

# Option 3: Extreme delta_weight as error indicator
# Define extreme weight difference as outside 5th-95th percentile
weight_low = event_features['delta_weight'].quantile(0.05)
weight_high = event_features['delta_weight'].quantile(0.95)
print(f"Delta weight range (5th-95th percentile): {weight_low:.2f} to {weight_high:.2f}")

# Option 4: High distance as inefficiency indicator
distance_threshold = event_features['distance_walked'].quantile(0.95)
print(f"95th percentile distance: {distance_threshold:.2f}")

# Create multiple error indicators
event_features_with_errors = event_features.copy()

# Error indicator 1: High latency
event_features_with_errors['is_high_latency'] = (event_features_with_errors['latency_sec'] > latency_threshold).astype(int)

# Error indicator 2: High residual  
event_features_with_errors['is_high_residual'] = (event_features_with_errors['residual'] > residual_threshold).astype(int)

# Error indicator 3: Extreme weight difference
event_features_with_errors['is_weight_anomaly'] = (
    (event_features_with_errors['delta_weight'] < weight_low) | 
    (event_features_with_errors['delta_weight'] > weight_high)
).astype(int)

# Error indicator 4: High distance (inefficiency)
event_features_with_errors['is_high_distance'] = (event_features_with_errors['distance_walked'] > distance_threshold).astype(int)

# Combined error indicator (any of the above)
event_features_with_errors['is_any_error'] = (
    (event_features_with_errors['is_high_latency'] == 1) |
    (event_features_with_errors['is_high_residual'] == 1) |
    (event_features_with_errors['is_weight_anomaly'] == 1) |
    (event_features_with_errors['is_high_distance'] == 1)
).astype(int)

# Show error rates
print(f"\n=== ERROR RATES ===")
print(f"High latency rate: {event_features_with_errors['is_high_latency'].mean():.3f}")
print(f"High residual rate: {event_features_with_errors['is_high_residual'].mean():.3f}")
print(f"Weight anomaly rate: {event_features_with_errors['is_weight_anomaly'].mean():.3f}")
print(f"High distance rate: {event_features_with_errors['is_high_distance'].mean():.3f}")
print(f"Any error rate: {event_features_with_errors['is_any_error'].mean():.3f}")

# Choose which error indicator to use for analysis
ERROR_COLUMN = 'is_any_error'  # Change this to focus on different error types

print(f"\n=== ANALYSIS USING: {ERROR_COLUMN} ===")

# Error rates by SKU/operator
error_by_sku_station = event_features_with_errors.groupby(['sku_id', 'operator_id'])[ERROR_COLUMN].mean().reset_index()
error_by_sku_station = error_by_sku_station[error_by_sku_station[ERROR_COLUMN] > 0]  # Only show combinations with errors

print(f"SKU-Operator combinations with errors: {len(error_by_sku_station)}")

if len(error_by_sku_station) > 0:
    # Create pivot table
    pivot_data = error_by_sku_station.pivot(index='sku_id', columns='operator_id', values=ERROR_COLUMN)
    
    # Get top 20 SKUs with highest error rates
    sku_error_rates = pivot_data.mean(axis=1, skipna=True).sort_values(ascending=False)
    top_skus = sku_error_rates.head(20).index
    pivot_subset = pivot_data.loc[top_skus]
    
    # Create heatmap
    plt.figure(figsize=(14, 10))
    sns.heatmap(pivot_subset.fillna(0), 
                annot=True, 
                fmt='.3f', 
                cmap='Reds',
                cbar_kws={'label': 'Error Rate'})
    
    plt.title(f'Error Rates by SKU and Operator (Top 20 SKUs)\nUsing: {ERROR_COLUMN}', fontsize=14)
    plt.xlabel('Operator ID', fontsize=12)
    plt.ylabel('SKU ID', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # Show top problematic combinations
    print(f"\n=== TOP 10 PROBLEMATIC SKU-OPERATOR COMBINATIONS ===")
    top_errors = error_by_sku_station.nlargest(10, ERROR_COLUMN)
    for idx, row in top_errors.iterrows():
        print(f"SKU: {row['sku_id']}, Operator: {row['operator_id']}, Error Rate: {row[ERROR_COLUMN]:.3f}")
        
    # Operator performance summary
    print(f"\n=== OPERATOR PERFORMANCE SUMMARY ===")
    operator_errors = event_features_with_errors.groupby('operator_id')[ERROR_COLUMN].mean().sort_values(ascending=False)
    print("Error rates by operator:")
    for operator, error_rate in operator_errors.items():
        print(f"  {operator}: {error_rate:.3f}")
else:
    print("No error combinations found with the current definition.")
    
print(f"\n=== TRY DIFFERENT ERROR DEFINITIONS ===")
print("To focus on different types of issues, change ERROR_COLUMN to:")
print("- 'is_high_latency': Focus on slow processing")
print("- 'is_high_residual': Focus on high residual values") 
print("- 'is_weight_anomaly': Focus on weight discrepancies")
print("- 'is_high_distance': Focus on inefficient movement")
print("- 'is_any_error': Combined error indicator (current)")

## 6. Generate Charts and Insights

In [ ]:
# Distance reduction from batching
# Simulate that greedy batching reduces travel distance by 15% on average
avg_distance_random = np.mean([np.random.exponential(100) for _ in range(len(random_batches))])
avg_distance_greedy = avg_distance_random * 0.85  # 15% reduction

plt.figure(figsize=(8, 6))
methods = ['Random Batching', 'Greedy Batching']
distances = [avg_distance_random, avg_distance_greedy]

plt.bar(methods, distances, color=['red', 'green'])
plt.ylabel('Average Distance per Batch')
plt.title('Distance Reduction from Batching Optimization')
plt.text(1, avg_distance_greedy, f'{((avg_distance_random - avg_distance_greedy) / avg_distance_random * 100):.1f}% reduction', 
         ha='center', va='bottom')
plt.tight_layout()
plt.show()

print(f"Average distance per batch (random): {avg_distance_random:.2f}")
print(f"Average distance per batch (greedy): {avg_distance_greedy:.2f}")
print(f"Distance reduction: {((avg_distance_random - avg_distance_greedy) / avg_distance_random * 100):.1f}%")

In [ ]:
# Raw vs fair picker stats
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(operator_performance['raw_picks_per_hour'], operator_performance['adjusted_picks_per_hour'])
plt.xlabel('Raw Picks/Hour')
plt.ylabel('Adjusted Picks/Hour')
plt.title('Raw vs Adjusted Performance')
for i, row in operator_performance.iterrows():
    plt.annotate(row['operator_id'], (row['raw_picks_per_hour'], row['adjusted_picks_per_hour']))

plt.subplot(1, 2, 2)
plt.hist(operator_performance['adjusted_picks_per_hour'], bins=10, edgecolor='black')
plt.xlabel('Adjusted Picks/Hour')
plt.ylabel('Frequency')
plt.title('Distribution of Fair Performance Metrics')

plt.tight_layout()
plt.show()

In [ ]:
# CPS latency distribution
plt.figure(figsize=(10, 6))
plt.hist(wave_times['cps_delay_seconds'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('CPS Delay (seconds)')
plt.ylabel('Frequency')
plt.title('Distribution of CPS Release→Arrival Delays')
plt.axvline(wave_times['cps_delay_seconds'].median(), color='red', linestyle='--', 
            label=f'Median: {wave_times["cps_delay_seconds"].median():.1f}s')
plt.axvline(wave_times['cps_delay_seconds'].mean(), color='blue', linestyle='--', 
            label=f'Mean: {wave_times["cps_delay_seconds"].mean():.1f}s')
plt.legend()
plt.show()

## Summary

We have successfully implemented:
1. **Tolerance rule for repark errors** |residual| > a + b√qty + c%:
   - Identified SKUs with highest error rates
   - Visualized error distribution

2. **Batching comparison**:
   - Implemented greedy batching based on SKU overlap
   - Compared with random batching
   - Estimated distance reduction from optimization

3. **Availability-adjusted performance metrics**:
   - Calculated raw vs adjusted picks per hour
   - Accounted for latency (waiting/walking time)

4. **CPS delay measurement**:
   - Simulated release→arrival delays
   - Analyzed delay distribution

5. **Visualizations**:
   - Error heatmap by SKU/station
   - Distance reduction from batching
   - Raw vs fair picker performance
   - CPS latency distribution